# 06. `sleep_duration` 회귀 기반 정밀 대치

지금까지(03/04)는 `sleep_duration` 결측을 `sleep_quality`별 조건부 중앙값 하나로만 채웠음(poor→6.45, average→6.99, good→7.54) — 같은 `sleep_quality`를 가진 사람은 전부 똑같은 값으로 뭉개짐. 이번엔 `bmi`, `exercise_duration`, `step_count`, `calorie_expenditure`, `water_intake`, `heart_rate` 등 다른 수치형 피처까지 같이 써서 **회귀모델로 연속값을 예측**해 6/7 경계 근처 애매한 케이스를 더 세밀하게 나눠봅니다.

`05_stress_level_imputation`에서 같은 접근이 실패했으므로(신호가 baseline 대비 +3%p뿐), **여기서도 먼저 회귀모델이 단순 조건부 중앙값보다 실제로 더 정확한지(RMSE 비교) 검증한 뒤에** 메인 모델에 반영할지 결정합니다.

커널: **Python (teammate)**

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import balanced_accuracy_score, accuracy_score, mean_squared_error
import lightgbm as lgb

SEED = 42
N_FOLDS = 5
DATA_DIR = Path("../playground-series-s6e7")
OUT_DIR = DATA_DIR / "processed"
TARGET = "health_condition"

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
folds = pd.read_csv(OUT_DIR / "cv_folds.csv")
train = train.merge(folds, on="id", how="left")
assert train["fold"].isna().sum() == 0

print("sleep_duration 결측:", train["sleep_duration"].isna().sum(), f"({train['sleep_duration'].isna().mean()*100:.1f}%)")

sleep_duration 결측: 75999 (11.0%)


## 1. 04와 동일한 결측 복구 피처 (baseline 재사용, physical_activity만)

In [2]:
def add_activity_recovery(df, step_tertiles):
    df = df.copy()
    activity_proxy = pd.cut(
        df["step_count"], bins=[-np.inf, step_tertiles[0], step_tertiles[1], np.inf],
        labels=["sedentary", "moderate", "active"],
    ).astype(object)
    df["physical_activity_level_recovered"] = df["physical_activity_level"]
    missing_activity = df["physical_activity_level"].isna()
    df.loc[missing_activity, "physical_activity_level_recovered"] = activity_proxy[missing_activity]
    df["physical_activity_level_recovered"] = df["physical_activity_level_recovered"].fillna("moderate")

    df["stress_level_isnull"] = df["stress_level"].isna().astype(np.int8)
    df["sleep_duration_isnull"] = df["sleep_duration"].isna().astype(np.int8)
    df["physical_activity_level_isnull"] = df["physical_activity_level"].isna().astype(np.int8)
    return df


step_tertiles = train["step_count"].quantile([1/3, 2/3]).values
train = add_activity_recovery(train, step_tertiles)
test = add_activity_recovery(test, step_tertiles)

# 기존(03/04) 방식: sleep_quality 조건부 중앙값 -- 비교 기준선으로 유지
sleep_quality_cond = train.groupby("sleep_quality")["sleep_duration"].median().to_dict()
sleep_quality_cond["missing"] = train["sleep_duration"].median()
print("기존 조건부 중앙값:", sleep_quality_cond)

기존 조건부 중앙값: {'average': 6.99, 'good': 7.54, 'poor': 6.45, 'missing': np.float64(6.99)}


## 2. 보조 회귀모델용 피처셋 구성 (원본 train/test는 건드리지 않음)

In [3]:
AUX_NUMERIC = ["heart_rate", "bmi", "calorie_expenditure", "step_count", "exercise_duration", "water_intake"]
AUX_ORDINAL = {
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level_recovered": ["sedentary", "moderate", "active"],
    "smoking_alcohol": ["no", "occasional", "yes"],
    "stress_level": ["low", "medium", "high"],
}
AUX_NOMINAL = ["diet_type", "gender"]
AUX_FLAGS = ["stress_level_isnull", "physical_activity_level_isnull"]

aux_train_raw = train[AUX_NUMERIC + list(AUX_ORDINAL.keys()) + AUX_NOMINAL + AUX_FLAGS].copy()
aux_test_raw = test[AUX_NUMERIC + list(AUX_ORDINAL.keys()) + AUX_NOMINAL + AUX_FLAGS].copy()

aux_numeric_medians = aux_train_raw[AUX_NUMERIC].median()
for df in (aux_train_raw, aux_test_raw):
    for col in AUX_NUMERIC:
        df[col] = df[col].fillna(aux_numeric_medians[col])

aux_categorical_cols = list(AUX_ORDINAL.keys()) + AUX_NOMINAL
for df in (aux_train_raw, aux_test_raw):
    for col in aux_categorical_cols:
        df[col] = df[col].fillna("missing")

for col, order in AUX_ORDINAL.items():
    categories = order + ["missing"]
    enc = OrdinalEncoder(categories=[categories])
    aux_train_raw[col] = enc.fit_transform(aux_train_raw[[col]])
    aux_test_raw[col] = enc.transform(aux_test_raw[[col]])

aux_train_ohe = pd.get_dummies(aux_train_raw[AUX_NOMINAL], prefix=[f"aux_{c}" for c in AUX_NOMINAL])
aux_test_ohe = pd.get_dummies(aux_test_raw[AUX_NOMINAL], prefix=[f"aux_{c}" for c in AUX_NOMINAL])
aux_test_ohe = aux_test_ohe.reindex(columns=aux_train_ohe.columns, fill_value=0)

AUX_FEATURE_COLS = AUX_NUMERIC + list(AUX_ORDINAL.keys()) + AUX_FLAGS + list(aux_train_ohe.columns)

aux_train_X = pd.concat([aux_train_raw[AUX_NUMERIC + list(AUX_ORDINAL.keys()) + AUX_FLAGS], aux_train_ohe], axis=1)
aux_test_X = pd.concat([aux_test_raw[AUX_NUMERIC + list(AUX_ORDINAL.keys()) + AUX_FLAGS], aux_test_ohe], axis=1)

print(len(AUX_FEATURE_COLS), "aux features")

20 aux features


## 3. OOF로 보조 회귀모델 학습 + 기존 방식과 RMSE 비교

`sleep_duration`이 관측된 행만 대상으로, fold별 held-out에서 (a) 회귀모델 예측 vs (b) 기존 조건부 중앙값의 RMSE를 비교합니다.

In [4]:
sleep_observed = train["sleep_duration"].notna()
oof_sleep_pred = np.full(len(train), np.nan)

aux_params = dict(
    objective="regression", metric="rmse", n_estimators=500,
    learning_rate=0.05, num_leaves=63, subsample=0.8,
    colsample_bytree=0.8, random_state=SEED, verbosity=-1,
)

fold_scores = []
for fold in range(N_FOLDS):
    aux_tr_mask = (train["fold"] != fold) & sleep_observed
    X_tr = aux_train_X.loc[aux_tr_mask, AUX_FEATURE_COLS]
    y_tr = train.loc[aux_tr_mask, "sleep_duration"]

    val_mask = (train["fold"] == fold) & sleep_observed
    X_val = aux_train_X.loc[val_mask, AUX_FEATURE_COLS]
    y_val = train.loc[val_mask, "sleep_duration"]

    model = lgb.LGBMRegressor(**aux_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])

    val_pred = model.predict(X_val)
    rmse_model = mean_squared_error(y_val, val_pred) ** 0.5

    # 기존 방식: sleep_quality 조건부 중앙값 (해당 fold를 뺀 나머지로 다시 계산해야 공정하지만,
    # 경계값이 폴드 간 거의 안 변하므로 04에서 쓴 값 그대로 비교)
    baseline_pred = train.loc[val_mask, "sleep_quality"].fillna("missing").map(sleep_quality_cond)
    rmse_baseline = mean_squared_error(y_val, baseline_pred) ** 0.5

    fold_scores.append((rmse_model, rmse_baseline))
    print(f"fold {fold}: RMSE 회귀모델={rmse_model:.4f}  RMSE 기존(조건부중앙값)={rmse_baseline:.4f}")

    fold_all_mask = train["fold"] == fold
    oof_sleep_pred[fold_all_mask.values] = model.predict(aux_train_X.loc[fold_all_mask, AUX_FEATURE_COLS])

avg_model = np.mean([s[0] for s in fold_scores])
avg_baseline = np.mean([s[1] for s in fold_scores])
print(f"\n평균 RMSE - 회귀모델: {avg_model:.4f} / 기존 조건부중앙값: {avg_baseline:.4f}")
print(f"개선폭: {avg_baseline - avg_model:+.4f} (양수면 회귀모델이 더 정확함)")

fold 0: RMSE 회귀모델=1.1198  RMSE 기존(조건부중앙값)=1.1350


fold 1: RMSE 회귀모델=1.1193  RMSE 기존(조건부중앙값)=1.1333


fold 2: RMSE 회귀모델=1.1182  RMSE 기존(조건부중앙값)=1.1330


fold 3: RMSE 회귀모델=1.1185  RMSE 기존(조건부중앙값)=1.1329


fold 4: RMSE 회귀모델=1.1230  RMSE 기존(조건부중앙값)=1.1371

평균 RMSE - 회귀모델: 1.1198 / 기존 조건부중앙값: 1.1342
개선폭: +0.0145 (양수면 회귀모델이 더 정확함)


## 4. 6/7 경계 판정 정확도까지 확인

RMSE가 조금 좋아져도 실제로 중요한 건 "6 미만인지", "7 이상인지"를 얼마나 잘 맞히는가입니다 (라벨 생성 규칙의 핵심 임계값).

In [5]:
obs_idx = train.index[sleep_observed]
true_vals = train.loc[obs_idx, "sleep_duration"]
model_preds = pd.Series(oof_sleep_pred[obs_idx], index=obs_idx)
baseline_preds = train.loc[obs_idx, "sleep_quality"].fillna("missing").map(sleep_quality_cond)

def band(x):
    return np.where(x < 6, "lt6", np.where(x >= 7, "ge7", "mid"))

true_band = band(true_vals)
model_band_acc = (band(model_preds) == true_band).mean()
baseline_band_acc = (band(baseline_preds) == true_band).mean()

print(f"6/7 구간 판정 정확도 - 회귀모델: {model_band_acc:.4f} / 기존 조건부중앙값: {baseline_band_acc:.4f}")

6/7 구간 판정 정확도 - 회귀모델: 0.4938 / 기존 조건부중앙값: 0.4279


## 5. 개선 확인되면 메인 모델에 반영 (04와 동일 구조 + sleep_duration_recovered만 교체)

In [6]:
use_regression = avg_model < avg_baseline
print("채택 여부:", "회귀모델 채택" if use_regression else "기존 조건부중앙값 유지 (회귀모델이 더 나쁨)")

if use_regression:
    train["sleep_duration_recovered"] = train["sleep_duration"]
    train.loc[~sleep_observed, "sleep_duration_recovered"] = oof_sleep_pred[~sleep_observed.values]

    final_aux_model = lgb.LGBMRegressor(**aux_params)
    final_aux_model.fit(aux_train_X.loc[sleep_observed, AUX_FEATURE_COLS], train.loc[sleep_observed, "sleep_duration"])

    test_sleep_observed = test["sleep_duration"].notna()
    test["sleep_duration_recovered"] = test["sleep_duration"]
    test_missing_idx = test.index[~test_sleep_observed]
    test.loc[test_missing_idx, "sleep_duration_recovered"] = final_aux_model.predict(
        aux_test_X.loc[test_missing_idx, AUX_FEATURE_COLS]
    )
else:
    train["sleep_duration_recovered"] = train["sleep_duration"]
    missing_sleep = train["sleep_duration"].isna()
    train.loc[missing_sleep, "sleep_duration_recovered"] = train.loc[missing_sleep, "sleep_quality"].fillna("missing").map(sleep_quality_cond)

    test["sleep_duration_recovered"] = test["sleep_duration"]
    missing_sleep_test = test["sleep_duration"].isna()
    test.loc[missing_sleep_test, "sleep_duration_recovered"] = test.loc[missing_sleep_test, "sleep_quality"].fillna("missing").map(sleep_quality_cond)

채택 여부: 회귀모델 채택


In [7]:
NUMERIC_COLS = [
    "sleep_duration_recovered", "heart_rate", "bmi", "calorie_expenditure",
    "step_count", "exercise_duration", "water_intake",
]
ORDINAL_COLS = {
    "stress_level": ["low", "medium", "high"],
    "sleep_quality": ["poor", "average", "good"],
    "physical_activity_level_recovered": ["sedentary", "moderate", "active"],
    "smoking_alcohol": ["no", "occasional", "yes"],
}
NOMINAL_COLS = ["diet_type", "gender"]
FLAG_COLS = ["stress_level_isnull", "sleep_duration_isnull", "physical_activity_level_isnull"]

numeric_medians = train[NUMERIC_COLS].median()
for df in (train, test):
    for col in NUMERIC_COLS:
        df[col] = df[col].fillna(numeric_medians[col])

categorical_cols = list(ORDINAL_COLS.keys()) + NOMINAL_COLS
for df in (train, test):
    for col in categorical_cols:
        df[col] = df[col].fillna("missing")

for col, order in ORDINAL_COLS.items():
    categories = order + ["missing"]
    encoder = OrdinalEncoder(categories=[categories])
    train[col] = encoder.fit_transform(train[[col]])
    test[col] = encoder.transform(test[[col]])

train_ohe = pd.get_dummies(train[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = pd.get_dummies(test[NOMINAL_COLS], prefix=NOMINAL_COLS)
test_ohe = test_ohe.reindex(columns=train_ohe.columns, fill_value=0)
train = pd.concat([train.drop(columns=NOMINAL_COLS), train_ohe], axis=1)
test = pd.concat([test.drop(columns=NOMINAL_COLS), test_ohe], axis=1)

FEATURE_COLS_V3 = NUMERIC_COLS + list(ORDINAL_COLS.keys()) + FLAG_COLS + list(train_ohe.columns)

target_encoder = LabelEncoder()
train["target_enc"] = target_encoder.fit_transform(train[TARGET])
lgb_class_order = list(target_encoder.classes_)
train_priors = train[TARGET].value_counts(normalize=True).to_dict()

print(len(FEATURE_COLS_V3), "features")

22 features


In [8]:
def prior_corrected_predict(proba, class_order, priors):
    prior_arr = np.array([priors[c] for c in class_order])
    scores = proba / prior_arr
    return np.array(class_order)[scores.argmax(axis=1)]


main_params = dict(
    objective="multiclass", num_class=3, n_estimators=500,
    learning_rate=0.05, num_leaves=63, subsample=0.8,
    colsample_bytree=0.8, random_state=SEED, verbosity=-1,
)

oof_proba_v3 = np.zeros((len(train), 3))
for fold in range(N_FOLDS):
    tr_idx = train["fold"] != fold
    va_idx = train["fold"] == fold
    X_tr, y_tr = train.loc[tr_idx, FEATURE_COLS_V3], train.loc[tr_idx, "target_enc"]
    X_va, y_va = train.loc[va_idx, FEATURE_COLS_V3], train.loc[va_idx, "target_enc"]

    model = lgb.LGBMClassifier(**main_params)
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    oof_proba_v3[va_idx.values] = model.predict_proba(X_va)
    print(f"fold {fold} done")

pred_v3 = prior_corrected_predict(oof_proba_v3, lgb_class_order, train_priors)
ba_v3 = balanced_accuracy_score(train[TARGET].values, pred_v3)
acc_v3 = accuracy_score(train[TARGET].values, pred_v3)

print(f"\n04 baseline(사전확률 보정, 튜닝)  : 0.94987")
print(f"06 sleep_duration 회귀 대치 적용   : {ba_v3:.5f}  (accuracy {acc_v3:.5f})")
print(f"개선폭 (04 대비)                   : {ba_v3 - 0.94987:+.5f}")

fold 0 done


fold 1 done


fold 2 done


fold 3 done


fold 4 done



04 baseline(사전확률 보정, 튜닝)  : 0.94987
06 sleep_duration 회귀 대치 적용   : 0.94974  (accuracy 0.93918)
개선폭 (04 대비)                   : -0.00013


## 6. sleep_duration 결측 행만 따로 본 BA + 최종 제출 파일

In [9]:
sleep_missing_mask = train["sleep_duration_isnull"] == 1
ba_sleep_missing_v3 = balanced_accuracy_score(train.loc[sleep_missing_mask, TARGET], pred_v3[sleep_missing_mask.values])
print(f"sleep_duration 결측 행({sleep_missing_mask.sum()}개)만 따로 본 BA: {ba_sleep_missing_v3:.5f}")

if ba_v3 > 0.94987:
    final_model = lgb.LGBMClassifier(**main_params)
    final_model.fit(train[FEATURE_COLS_V3], train["target_enc"])
    test_proba = final_model.predict_proba(test[FEATURE_COLS_V3])
    test_pred = prior_corrected_predict(test_proba, lgb_class_order, train_priors)
    submission = pd.DataFrame({"id": test["id"], TARGET: test_pred})
    submission.to_csv(OUT_DIR / "submission_v4_sleep_regression.csv", index=False)
    print("개선 확인, 저장:", OUT_DIR / "submission_v4_sleep_regression.csv")
    print(submission[TARGET].value_counts(normalize=True))
else:
    print("04 대비 개선 없음 -> submission_v2_tuned.csv를 그대로 유지")

sleep_duration 결측 행(75999개)만 따로 본 BA: 0.86094
04 대비 개선 없음 -> submission_v2_tuned.csv를 그대로 유지
